### **Packages**

In [ ]:
library(survey)
library(dplyr)
library(haven)

### **Data**

In [99]:
raw.data <- read_sav("../Vaccine_dropout_mz/Data/IDS_2022_MZKR81FL.SAV")

### **Measles I Recode**

In [100]:
# Measles vaccination status Based on health card recorde with date
raw.data <- raw.data |> 
		mutate(zero_dose = car::recode(H9, 'c(1, 2, 3) = 0;
	                             c(0,8) = 1'))

count(raw.data, zero_dose)

# A tibble: 3 × 2
  zero_dose                         n
  <dbl+lbl>                     <int>
1  0 [No]                        2722
2  1 [Vaccination date on card]  2668
3 NA                             3899

### **Recode variables**

In [101]:
recode_variables <- function(ids_data){
  
  ids_data <-  ids_data |>
  mutate(mother_age = car::recode(as.numeric(V013),
                                  "c(1,2)='15-24'; 
                                   c(3,4)='25-34'; 
                                   c(5,6,7,8,9)='≥35'"),
         
  mother_marital_status = car::recode(as.numeric(V501),
                                      "0='Single/Never in union';
                                       c(1,2)='Married/Cohabitation';
                                       3:5='Divorced/Separated/Widowed'"),
  
  mother_education = car::recode(as.numeric(V106),
                                 "0='Illiterate';
                                 1='Primary';
                                 2:3='Secondary/above'"),
  
  mother_occupation = car::recode(as.numeric(V714),
                                  "0='Unemployed';
                                  1='Employed'"),
  
  antenatal_visits = car::recode(as.numeric(M14), 
                                 "0='No visits';
                                 c(1,2,3)='1 to 3'; 
                                 4:15='≥4'"),

  birth_order = car::recode(as.numeric(BORD), 
                            "1='1'; 
                            c(2,3)='2 to 3'; 
                            4:13='≥4' "),
  
  place_of_delivery = car::recode(as.numeric(M15), 
                                  "1:13='At home/Other'; 
                                  14:96='Institutional'"),
  
  has_a_health_card = car::recode(as.numeric(H1), 
                                  "c(0,3)='No'; 
                                  1='Yes seen'; 
                                  2='Yes not seen' "),

  number_of_household_members = car::recode(as.numeric(V136),
                                            "c(2,3,4)='2 to 4';
                                            5:23='≥5'"),
  
  number_of_children_u5h = car::recode(as.numeric(V137), 
                                                            "c(1, 2)='1 to 2';
                                                            3:9='≥3' "), 
  
  distance_to_health_facility = car::recode(as.numeric(V467D), 
                                            "0='No problem';
                                            1='Big problem'; 
                                            2='Not a big problem'"),
  
  religion = car::recode(as.numeric(V130), 
                         "1='Catholic'; 
                         2='Islamic'; 
                         3:5='Protestants';
                         c(6)='Others' "),
  
  province = car::recode(as.numeric(V024), 
                       "1='Niassa'; 
                        2='Cabo Delgado'; 
                        3='Nampula'; 
                        4='Zambézia'; 
                        5='Tete'; 
                        6='Manica'; 
                        7='Sofala'; 
                        8='Inhambane'; 
                        9='Gaza'; 
                        10='Maputo Provincia'; 
                        11='Cidade de Maputo'"),
  
  sex_of_child = as_factor(B4),
  area_of_residence = as_factor(V025),
  wealth_index = as_factor(V190)
  
  ) 
  



# Adicionando a variavel regiao
ids_data$region[ids_data$V024 < 4] <- "Northern"
ids_data$region[ids_data$V024 > 3 & ids_data$V024 < 8] <- "Central"
ids_data$region[ids_data$V024 > 7] <- "Southern"

# Adicionando Idade da crianca
ids_data$age <- ids_data$V008-ids_data$B3

# Recode de Ano
ids_data$year <- 2022


# Ordenando as variaveis
ids_data$mother_age <- factor(ids_data$mother_age, 
                              level=c('15-24', 
                                      '25-34', 
                                      '≥35'))

ids_data$mother_marital_status <- factor(ids_data$mother_marital_status, 
                                         c("Single/Never in union",
                                           "Married/Cohabitation",  
                                           "Divorced/Separated/Widowed"))

ids_data$mother_education <- factor(ids_data$mother_education, 
                                    levels = c("Illiterate", 
                                               "Primary", 
                                               "Secondary/above"))

ids_data$mother_occupation <- factor(ids_data$mother_occupation, 
                                     levels = c('Unemployed', 
                                                'Employed'))

ids_data$antenatal_visits <- factor(ids_data$antenatal_visits, 
                                    levels = c("No visits",
                                               "1 to 3",
                                               "≥4"))

ids_data$place_of_delivery <- factor(ids_data$place_of_delivery, 
                                     levels = c("At home/Other", 
                                                "Institutional"))

ids_data$birth_order <- factor(ids_data$birth_order, 
                               levels = c("1",
                                          "2 to 3",
                                          "≥4"))

ids_data$has_a_health_card <- factor(ids_data$has_a_health_card, 
                                     levels = c("No", 
                                                "Yes seen", 
                                                "Yes not seen"))

ids_data$religion <- factor(ids_data$religion, 
                            levels = c("Protestants",
                                       "Catholic", 
                                       "Islamic",
                                       "Others"))

ids_data$region <- factor(ids_data$region, 
                          levels = c("Southern",
                                     "Central", 
                                     "Northern"))

ids_data$number_of_children_u5h <- factor(ids_data$number_of_children_u5h, 
                                                               levels = c("1 to 2",
                                                                          "≥3"))

ids_data$number_of_household_members <- factor(ids_data$number_of_household_members,
                                               levels = c( "2 to 4", 
                                                          "≥5"))

ids_data$distance_to_health_facility <- factor(ids_data$distance_to_health_facility,
                                               levels = c(#"No problem", 
                                                          "Big problem", 
                                                          "Not a big problem"))

ids_data$province <- factor(ids_data$province, 
                            levels = c("Cidade de Maputo",
                                       "Niassa", 
                                       "Cabo Delgado", 
                                       "Nampula", 
                                       "Zambézia",
                                       "Tete", 
                                       "Manica",
                                       "Sofala", 
                                       "Inhambane", 
                                       "Gaza", 
                                       "Maputo Provincia"))


return(ids_data)

}


In [ ]:
# call
raw.data_2 <- recode_variables(raw.data)

### **Weighted measles ZD estimates for 12-23 children**

In [103]:
# survey object - simple random sampling
raw.data_2$wt <- raw.data_2$V005/1e6
raw.data_2$i <- 1

data_wt <- svydesign(ids = ~V021, weights = ~wt, strata = ~V022, data = raw.data_2)

data_wt_23 <- subset(data_wt, B19 >= 12 & 
                                 B19 < 24 &
                                 B5 == 1)

# Estimates
zd <- svyciprop(formula = ~zero_dose, 
	method = "logit",
	design = data_wt_23) 

In [104]:
paste0(round(as.numeric(zd)*100,2), "% [", round(attr(zd, "ci")[1] *100, 1), "-", round(attr(zd, "ci")[2] * 100, 2),"]")

[1] "38.28% [34.9-41.77]"

### **Sociodemographics and Prevalence**

In [10]:
variaveis <- c("mother_age", 
               "mother_education",
               "mother_marital_status", 
               "mother_occupation",
               "wealth_index",
               "religion", 
               "antenatal_visits", 
               "place_of_delivery",
               "distance_to_health_facility", 
               "has_a_health_card",
               "birth_order",
               "sex_of_child",
               "number_of_household_members", 
               "number_of_children_u5h", 
               "area_of_residence",
               "province",
               "region"
               )

In [11]:
make_props_table <- function(variaveis, dados){
  # table list
  tbl_list <- list()
  # Loop through all variables
  for (i in variaveis) {
    # Totals estimation
    tabl <- svytotal(as.formula(paste0("~", i)),na.rm=T, dados) |> 
      as.data.frame() # save as dataframe
    # Table formatation and CI95 caculation  
    tabl <- tabl |>
            mutate(
              `N%`= paste0(round(total), " (", round(total/sum(total)*100,1),")"),
              `CI95`= paste0(round(total - 1.96*SE, 1), "-", round(total + 1.96 *SE,1)),
              Characteristic= stringr::str_remove(rownames(tabl), i)) |>
            select(Characteristic, `N%`, `CI95`)
    
    rownames(tabl) <- NULL #drop rownames
    # final format of table
    formated_table <- rbind(c(i,rep("",2)), tabl)
    tbl_list[[i]] <- formated_table # save each variable total estimation
  }
  
    return(bind_rows(tbl_list)) # join all tables # outside loop
  }

# Generate and export table
prop_table <- make_props_table(variaveis, data_wt_23)
openxlsx::write.xlsx(prop_table, "./Tabela_1_caracteristicas_sociodemograficas_zd.xlsx")
prop_table

                Characteristic          N%          CI95
1                   mother_age                          
2                        15-24  851 (47.1)     759-942.8
3                        25-34  646 (35.8)   578.2-714.4
4                          ≥35  310 (17.2)   263.9-356.7
5             mother_education                          
6                   Illiterate  533 (29.5)   462.9-602.5
7                      Primary  879 (48.6)   779.8-977.5
8              Secondary/above  396 (21.9)   346.3-446.1
9        mother_marital_status                          
10       Single/Never in union    98 (5.4)    72.7-123.9
11        Married/Cohabitation 1497 (82.8)   1386-1608.3
12  Divorced/Separated/Widowed  212 (11.7)   168.9-255.1
13           mother_occupation                          
14                  Unemployed 1325 (73.3) 1210.7-1439.5
15                    Employed  482 (26.7)   426.5-538.3
16                wealth_index                          
17                     Poorest 

In [12]:
svytotal(x=~i, design = data_wt_23)

   total     SE
i 1807.5 65.681

In [13]:
prev_list <- list()

make_prev_table <- function(outcome_variable ,variaveis, survey_data){
  for(category in variaveis){
    cat(category, "😜✔️\n")
    # table of props
    table <- svyby(formula = as.formula(paste0("~",outcome_variable)), 
                   by= as.formula(paste0("~",category)), 
                   design = survey_data, 
                   FUN = svyciprop , 
                   method = "beta",
                   vartype = "ci",
									 na.rm =T)
    # chi-square
    chi_square <- svychisq(formula = as.formula(paste0("~",outcome_variable,"+", category)),
                           design = survey_data)
    
    # joining prevalence and ci
    table <- table |>
      rename(Category = category, 
             Prevalence=outcome_variable) |>
      mutate(P_value = round(chi_square$p.value,3),
          Prevalence = paste0(round( Prevalence * 100, 1), 
                                " (", 
                                round(ci_l * 100, 1),
                                "-",
                                round(ci_u * 100, 1),
                                ")")) |>
      select(Category, Prevalence, P_value)
    
    # adding labels for each variable data
    edited_table <- rbind(" ", table)
    edited_table <- mutate(edited_table, 
                           Category = case_when(is.na(Category) ~ category,
                                                           TRUE ~ Category))
    # save table in a list object
    prev_list[[category]] <- edited_table
  
}
  # return a single dataframe
  return(bind_rows(prev_list))

}


In [15]:
prev_tab_zd <-  make_prev_table(outcome_variable = "zero_dose",
                                        variaveis = variaveis,
                                        survey_data = data_wt_23)
openxlsx::write.xlsx(prev_tab_zd, "./Tabela_2_Prevalencia_zd.xlsx")

mother_age 😜✔️
mother_education 😜✔️
mother_marital_status 😜✔️
mother_occupation 😜✔️
wealth_index 😜✔️
religion 😜✔️
antenatal_visits 😜✔️
place_of_delivery 😜✔️
distance_to_health_facility 😜✔️
has_a_health_card 😜✔️
birth_order 😜✔️
sex_of_child 😜✔️
number_of_household_members 😜✔️
number_of_children_u5h 😜✔️
area_of_residence 😜✔️
province 😜✔️
region 😜✔️


There were 19 warnings (use warnings() to see them)


In [16]:
prev_tab_zd

                                              Category       Prevalence P_value
1...1                                       mother_age                         
15-24                                            15-24 38.6 (33.9-43.5)   0.107
25-34                                            25-34   35 (29.6-40.7)   0.107
≥35                                                ≥35 44.3 (37.1-51.7)   0.107
1...5                                 mother_education                         
Illiterate                                  Illiterate 52.5 (46.1-58.8)       0
Primary                                        Primary 38.3 (33.1-43.7)       0
Secondary/above                        Secondary/above 19.1 (14.7-24.3)       0
1...9                            mother_marital_status                         
Single/Never in union            Single/Never in union 31.2 (18.2-46.7)   0.369
Married/Cohabitation              Married/Cohabitation 38.1 (34.4-41.9)   0.369
Divorced/Separated/Widowed  Divorced/Sep

### **Measles Zero-dose Regional Distributon**

In [19]:
# MOV by provinces
regions <- svyby(formula = ~zero_dose,by = ~province, FUN=svyciprop, method="beta", na.rm=T, vartype = "ci", design = data_wt) |>
  as_data_frame() |>
  mutate(zd = zd * 100, 
	ci_l = ci_l * 100,
	ci_u = ci_u * 100)

openxlsx::write.xlsx(regions, "./provinces_zd.xlsx")

### **Fitting Binary Logistic Regression Model**

In [111]:
re.factor <- function(ids_data){
	
# Ordenando as variaveis
ids_data$mother_age <- factor(ids_data$mother_age, 
                              level=c('25-34',
																      '15-24',
																		  '≥35'))

ids_data$mother_marital_status <- factor(ids_data$mother_marital_status, 
                                         c( "Married/Cohabitation",
																					  "Divorced/Separated/Widowed",
																					   "Single/Never in union"))

ids_data$mother_education <- factor(ids_data$mother_education, 
                                    levels = c("Secondary/above",
																			         "Illiterate", 
                                               "Primary"))

ids_data$mother_occupation <- factor(ids_data$mother_occupation, 
                                     levels = c('Employed',
																								'Unemployed'))

ids_data$antenatal_visits <- factor(ids_data$antenatal_visits, 
                                    levels = c("≥4",
																			        "No visits",
                                               "1 to 3"))

ids_data$place_of_delivery <- factor(ids_data$place_of_delivery, 
                                     levels = c("At home/Other", 
                                                "Institutional"))

ids_data$birth_order <- factor(ids_data$birth_order, 
                               levels = c("≥4",
																          "1",
                                          "2 to 3"))

ids_data$has_a_health_card <- factor(ids_data$has_a_health_card, 
                                     levels = c("Yes seen",
																			          "Yes not seen",
																			          "No"))

ids_data$religion <- factor(ids_data$religion, 
                            levels = c("Protestants",
                                       "Catholic", 
                                       "Islamic",
                                       "Others"))
	
ids_data$wealth_index <- factor(ids_data$wealth_index,
																levels = c(
																	         "Richest",
															             "Richer",
																					 "Middle",
																					 "Poorer",
																					 "Poorest"))

ids_data$region <- factor(ids_data$region, 
                          levels = c("Southern",
                                     "Central", 
                                     "Northern"))

ids_data$number_of_children_u5h <- factor(ids_data$number_of_children_u5h, 
                                                               levels = c("≥3",
                                                                          "1 to 2"))

ids_data$number_of_household_members <- factor(ids_data$number_of_household_members,
                                               levels = c("≥5",
                                                          "2 to 4"))

ids_data$distance_to_health_facility <- factor(ids_data$distance_to_health_facility,
                                               levels = c("Big problem", 
                                                          "Not a big problem"))

ids_data$province <- factor(ids_data$province, 
                            levels = c(
															         "Maputo Provincia",
                                       "Cidade de Maputo",
																			 "Niassa", 
                                       "Cabo Delgado", 
                                       "Nampula", 
                                       "Zambézia",
                                       "Tete", 
                                       "Manica",
                                       "Sofala", 
                                       "Inhambane", 
                                       "Gaza"))
	return(ids_data)
}

In [112]:
raw.data_2 <- re.factor(raw.data_2)

data_wt <- svydesign(ids = ~V021, weights = ~wt, strata = ~V022, data = raw.data_2)

data_wt_23 <- subset(data_wt, B19 >= 12 & 
                                 B19 < 24 &
                                 B5 == 1)

In [91]:
variaveis.modelo <- c(
	            "mother_age", 
              "mother_education",
              #"mother_marital_status", 
              "mother_occupation",
              "wealth_index",
              "religion", 
              "antenatal_visits", 
							"has_a_health_card",
              "distance_to_health_facility", 
              #"place_of_delivery", 
              #"number_of_children_u5h", 
              "area_of_residence",
              "province"
							#"region"
							)

In [113]:
mdl <- svyglm(formula = as.formula(paste0("zero_dose~", paste0(variaveis.modelo, collapse = "+"))), design = data_wt_23, family = binomial(link = "logit"))
warnings()

Warning message:
In eval(family$initialize) : non-integer #successes in a binomial glm!


Warning message:
In eval(family$initialize) : non-integer #successes in a binomial glm!

In [115]:
epxrt <- gtsummary::tbl_regression(mdl, exponentiate=T)

In [116]:
gtsummary::as_hux_xlsx(epxrt, "modelo_zd.xlsx")